In [106]:
from syncode import Syncode

# 1. Define the grammar
Syncode allows user to define a grammar using a simple EBNF syntax adapted from Lark. The grammars for some common programming languages are defined in the `syncode/grammars` directory. One can also simply feed the grammar rules directly as a string of rules as shown below.

In [107]:
grammar = r"""
    start: "Operands: " operand " and " operand " \n " "Operator: " operator

    ?operand: /-?[0-9]{1,10}\.[0-9]{1,10}/
    ?operator: /[+\-*\/]/
"""

# 2. Load the Huggingface models
Syncode uses the Huggingface transformers library to load the models. Simply provide the HuggingFace ID of the model to be used. Users can select the mode of generation `original` or `grammar_mask` to generate code with or without grammar constraining.

In [108]:
model_name = "microsoft/phi-2"

# Load the unconstrained original model
llm = Syncode(model = model_name, mode='original', max_new_tokens=20)

# Load the Syncode augmented model
syn_llm = Syncode(model = model_name, mode='grammar_mask', grammar=grammar)

[2026-10-28 02:36:35,310-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16
[2026-10-28 02:36:35,602-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[2026-10-28 02:36:37,301-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16
[2026-10-28 02:36:37,477-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[2026-10-28 02:36:40,913-accelerate.big_modeling] - Some parameters are on the meta device because they were offloaded to the cpu.


# 3. Run the operation
We use our simple grammar and a basic function

In [109]:
# Define function 
import operator

operations = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
}

def run_syncode(textual_query):
    # I didn't check out of bounds and stuff... since idt that's in scope
    # If you wanted to do that, you can just check if the int overflows and return an error
    prompt = """Given a query, extract the operands and operator. 
               Some words for + are: add, plus, addition, adding, 
               Some words for - are: minus, subtracted, removed,
               Some words for * are: times, multiplied,
               Some works for / are: divided, over, 
               An example is User query: What is 3.21 plus 5.829? Output: Operands 3.21 and 5.829 \n Operator +
               An example is User query: What is -3.21 minus 5.829? Output: Operands -3.21 and 5.829 \n Operator -
               An example is User query: What is 3.21 times or multiplied by -5.829? Output: Operands 3.21 and -5.829 \n Operator *
               An example is User query: What is - 3.21 divided by -5.829? Output: Operands 3.21 and -5.829 \n Operator /""" 
    parsed_query = syn_llm.infer(prompt + textual_query)
    print(parsed_query)
    splitted = parsed_query[0].split(" ")
    operand1 = splitted[1]
    operand2 = splitted[3]
    operator = splitted[-1]
    final_result = operations[operator](float(operand1), float(operand2))
    return(operand1, operand2, operator, final_result)

In [110]:
run_syncode("What is -9.5 minus -2.25?")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


['Operands: -9.5 and -2.25 \n Operator: -']


('-9.5', '-2.25', '-', -7.25)

In [111]:
run_syncode("What is -9.5 plus -2.25?")

['Operands: -9.5 and -2.25 \n Operator: +']


('-9.5', '-2.25', '+', -11.75)

In [112]:
run_syncode("What is 32.27238923 over (-8.72?)")

['Operands: 32.27238923 and -8.72 \n Operator: /']


('32.27238923', '-8.72', '/', -3.7009620676605506)

In [113]:
run_syncode("Can you calculate what negative 32.2723898998779879823 divided by  negative 8.72 is?")

['Operands: -32.2723898998 and -8.72 \n Operator: /']


('-32.2723898998', '-8.72', '/', 3.7009621444724763)

In [119]:
run_syncode("Can you calculate what negative -(32.2723898998779879823) times by negative 8.72 is?")

['Operands: -32.2723898998 and -8.72 \n Operator: /']


('-32.2723898998', '-8.72', '/', 3.7009621444724763)

In [118]:
run_syncode("Can you calculate what negative 32.2723898998779879823 plus - 8.72 is?")

['Operands: -32.2723898998 and -8.72 \n Operator: +']


('-32.2723898998', '-8.72', '+', -40.992389899799996)

In [117]:
run_syncode("Can you calculate what negative 32.0000 minus - 0000.72 is?")

['Operands: -32.0000 and -0000.72 \n Operator: -']


('-32.0000', '-0000.72', '-', -31.28)